In [1]:
from pathlib import Path

In [2]:
base = Path(".")

In [3]:
GENDER = {
    "slt": "female",
    "clb": "female",
    "axb": "female",
    "eey": "female",
    "ljm": "female",
    "lnh": "female",
    "bdl": "male",
    "rms": "male",
    "jmk": "male",
    "awb": "male",
    "ksp": "male",
    "ahw": "male",
    "aup": "male",
    "fem": "male",
    "gka": "male",
    "rxr": "male",
    "slp": "male",
}

In [4]:
ACCENTS = {
    "jmk": "Canadian English",
    "awb": "Scottish English",
    "ksp": "Indian English",
    "ahw": "German",
    "aup": "Indian",
    "axb": "Indian",
    "fem": "German",
    "gka": "Indian",
    "rxr": "Israeli",
    "slp": "Indian",
}

In [5]:
MISSING = {
    "jmk": {
        "arctic_a0564": "Then it is as I said, Womble announced",
        "arctic_b0313": "The apron string loomed near.",
        "arctic_b0229": "I saw it all myself."
    }
}

In [6]:
def read_text(file: Path):
    text_pairs = {}
    with open(file) as f:
        for line in f.readlines():
            text_start = line.find('"')
            text_end = line.rfind('"')
            text = line[text_start + 1:text_end]
            text_id = line[1:text_start].strip()
            text_pairs[text_id] = text
    return text_pairs

In [7]:
import soundfile as sf
from datasets import Dataset, Audio, Features, Value


/home/joregan/miniconda3/envs/hfnew/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
examples = []
for dir in base.glob("cmu_us_*"):
    speaker_id = dir.name.split("_")[-2]
    accent = ACCENTS.get(speaker_id, "American English")
    gender = GENDER.get(speaker_id, "unknown")
    text = dir / "etc" / "txt.done.data"
    pairs = read_text(text)
    for wav_file in dir.glob("wav/*.wav"):
        wav_id = wav_file.stem
        if wav_id not in pairs:
            print(f"Missing text for {wav_id} in {speaker_id}")
            continue
        text = pairs[wav_id]
        examples.append({
            "audio": str(wav_file.resolve()),
            "text": pairs[wav_id],
            "sentence_id": wav_id,
            "speaker_id": speaker_id,
            "gender": gender,
            "accent": accent,
        })

Missing text for arctic_a0564 in jmk
Missing text for arctic_b0313 in jmk
Missing text for arctic_b0229 in jmk
Missing text for arctic_a0512 in jmk
Missing text for arctic_a0576 in jmk
Missing text for arctic_a0456 in jmk
Missing text for arctic_a0542 in jmk
Missing text for arctic_a0392 in jmk
Missing text for arctic_b0223 in jmk
Missing text for arctic_a0130 in jmk
Missing text for arctic_a0108 in jmk
Missing text for arctic_a0578 in jmk
Missing text for arctic_a0568 in jmk
Missing text for arctic_a0341 in jmk
Missing text for arctic_a0561 in jmk
Missing text for arctic_a0575 in jmk
Missing text for arctic_a0208 in jmk
Missing text for arctic_a0565 in jmk
Missing text for arctic_a0507 in bdl
Missing text for arctic_a0282 in eey


```
Missing text for arctic_a0564 in jmk
Missing text for arctic_b0313 in jmk
Missing text for arctic_b0229 in jmk
Missing text for arctic_a0512 in jmk
Missing text for arctic_a0576 in jmk
Missing text for arctic_a0456 in jmk
Missing text for arctic_a0542 in jmk
Missing text for arctic_a0392 in jmk
Missing text for arctic_b0223 in jmk
Missing text for arctic_a0130 in jmk
Missing text for arctic_a0108 in jmk
Missing text for arctic_a0578 in jmk
Missing text for arctic_a0568 in jmk
Missing text for arctic_a0341 in jmk
Missing text for arctic_a0561 in jmk
Missing text for arctic_a0575 in jmk
Missing text for arctic_a0208 in jmk
Missing text for arctic_a0565 in jmk
Missing text for arctic_a0507 in bdl
Missing text for arctic_a0282 in eey
```

In [9]:
import pandas as pd
df = pd.DataFrame(examples)
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("audio", Audio("wav"))
dataset.save_to_disk(".")

Saving the dataset (4/4 shards): 100%|██████████| 15583/15583 [00:05<00:00, 2633.24 examples/s]
